In [1]:
import pandas as pd
import numpy as np
import shutil

In [2]:
excel_path = r"/home/david/Git/SDLT_Number_Crunching/Data/adiya_isla_scores.xlsx"
output_path = r"/home/david/Git/SDLT_Number_Crunching/Data/consolidated_results.xlsx"

# List of sheets to evaluate (skipping overview sheets like 'Whole' if present)
sheet_mapping = {
    "CoC": "Chain of Custody",
    "Au": "Authenticity",
    "In": "Integrity",
    "Re": "Reliability",
    "La": "Lawful Acquisition",
    "Fi": "Fact vs Inference",
    "Cq": "Competence and Qualification",
    "Rr": "Repeatability and reproducibility",
    "Pc": "Privacy compliance"
}

# ---------------------------------------------------------
# 2. Configuration Settings
# ---------------------------------------------------------
annotator_cols = ["annotator1", "annotator2", "annotator3"]
scale = ["0", "1", "2"]

In [3]:
# 1. Load the Whole sheet
whole_df = pd.read_excel(excel_path, sheet_name="Whole")
whole_df["Publication"] = whole_df["Publication"].astype(str).str.strip()

print(f"{'Publication':<40} | {'Column':<35} | {'Old Value':<10} | {'New Value':<10}")
print("-" * 105)

for sheet_name, column_name in sheet_mapping.items():
    sheet = pd.read_excel(excel_path, sheet_name=sheet_name, header=[0, 1])

    sheet.columns = pd.MultiIndex.from_tuples(
        [(str(c[0]).strip(), str(c[1]).strip()) for c in sheet.columns]
    )

    pub_col_tuple = [col for col in sheet.columns if col[0].lower() == "publication"][0]
    publications = sheet[pub_col_tuple].astype(str).str.strip()

    r2_annotators = [
        col for col in sheet["Round 2"].columns 
        if "annotator" in col.lower()
    ]

    round_2_data = sheet["Round 2"][r2_annotators].apply(pd.to_numeric, errors="coerce")
    medians = round_2_data.median(axis=1, skipna=True)

    median_map = pd.Series(medians.values, index=publications).dropna()

    target_col = next(
        (c for c in whole_df.columns if c.strip().lower() == column_name.lower()), 
        None
    )
    if target_col is None:
        print(f"Warning: Column '{column_name}' not found in 'Whole'. Skipping {sheet_name}.")
        continue

    new_values = whole_df["Publication"].map(median_map)
    current_numeric = pd.to_numeric(whole_df[target_col], errors="coerce")
    
    # Identify rows that actually need an update
    mismatch_mask = new_values.notna() & ~np.isclose(current_numeric, new_values, equal_nan=True)

    # Print changes before applying them
    if mismatch_mask.any():
        changed_rows = whole_df.loc[mismatch_mask, ["Publication", target_col]].copy()
        changed_rows["New_Value"] = new_values[mismatch_mask]

        for _, row in changed_rows.iterrows():
            pub = row["Publication"]
            old_val = row[target_col]
            new_val = row["New_Value"]
            print(f"{pub:<40} | {target_col:<35} | {str(old_val):<10} | {str(new_val):<10}")

        # Update whole_df
        whole_df.loc[mismatch_mask, target_col] = new_values[mismatch_mask]

# 2. Preserve all existing sheets by copying original file and replacing 'Whole'
shutil.copyfile(excel_path, output_path)

with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    whole_df.to_excel(writer, sheet_name="Whole", index=False)

print("-" * 105)
print("Consolidation complete.")

Publication                              | Column                              | Old Value  | New Value 
---------------------------------------------------------------------------------------------------------
Harding_2024                             | Chain of Custody                    | 1          | 2.0       
Hao_2018                                 | Chain of Custody                    | 1          | 0.0       
Frowis_2020                              | Chain of Custody                    | 2          | 1.0       
Harding_2024                             | Authenticity                        | 1          | 2.0       
Naha_2024                                | Authenticity                        | 0          | 1.0       
Ellul_2020                               | Integrity                           | 0          | 1.0       
Akbarfam_2023                            | Reliability                         | 1          | 2.0       
Harding_2024                             | Reliability

In [4]:
whole_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 10 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   Publication                        27 non-null     str  
 1   Chain of Custody                   27 non-null     int64
 2   Authenticity                       27 non-null     int64
 3   Integrity                          27 non-null     int64
 4   Reliability                        27 non-null     int64
 5   Lawful Acquisition                 27 non-null     int64
 6   Fact vs Inference                  27 non-null     int64
 7   Competence and Qualification       27 non-null     int64
 8   Repeatability and reproducibility  27 non-null     int64
 9   Privacy compliance                 27 non-null     int64
dtypes: int64(9), str(1)
memory usage: 2.2 KB
